# Chapter 5 Exercises: Loss Evaluation (Cross-Entropy & Perplexity)

Bu notebook, LLM'lerde **cross-entropy loss** ve **perplexity** metriklerini anlamak ve uygulamak için tasarlanmış egzersizler içerir.

## Egzersiz Konuları
1. Cross-entropy loss'un manuel hesaplanması
2. Perplexity hesaplama ve yorumlama
3. Training/validation loss hesaplama fonksiyonları
4. Pratik uygulamalar

## Gerekli Kütüphaneler ve Setup

In [1]:
import torch
import tiktoken
import math
from previous_chapters import GPTModel, create_dataloader_v1
# Alternatif:
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch02 import create_dataloader_v1

## Egzersiz 1: Manuel Cross-Entropy Hesaplama

**Hedef:** Cross-entropy loss'un nasıl hesaplandığını adım adım anlamak

Elimizde basit bir örnek var. Modelin çıktı logits'leri ve hedef token ID'leri:

In [ ]:
# Örnek veri
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves"
                       [40,    1107, 588]])   # ["I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you"
                        [1107,  588, 11311]]) # [" really like chocolate"]

### Adım 1: Modeli yükleyip logits alalım

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

with torch.no_grad():
    logits = model(inputs)

print(f"Logits shape: {logits.shape}")  # (batch_size, num_tokens, vocab_size)

### Egzersiz 1.1: Softmax Probability Hesaplama

**Görev:** Logits'leri softmax ile probability distribution'a çevirin

In [ ]:
# TODO: Softmax kullanarak probabilities hesapla
probas = torch.softmax(logits, dim=-1)

# Doğrulama
print(f"Probas shape: {probas.shape}")
# İlk batch'in ilk token için tüm olasılıkların toplamı 1 olmalı
print(f"Sum of probas for first token: {probas[0, 0].sum().item():.6f}")

### Egzersiz 1.2: Target Probability Hesaplama

**Görev:** Her hedef token için modelin öngördüğü olasılığı çıkarın

In [ ]:
# Her batch ve her pozisyon için hedef token olasılıklarını al
# probas[batch_idx, token_pos, target_token_id]

text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 1 target probabilities:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Text 2 target probabilities:", target_probas_2)

### Egzersiz 1.3: Log Probabilities Hesaplama

**Görev:** Target probabilities'ların log'unu alın ve yorumlayın

In [ ]:
# Log probabilities hesapla
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print("Log probabilities:", log_probas)

# Yorumlama:
# p = 1.0  ->  log(1.0) =  0.0   (Mükemmel tahmin)
# p = 0.5  ->  log(0.5) = -0.69
# p = 0.1  ->  log(0.1) = -2.30
# p = 0.001->  log(0.001)= -6.91
# p -> 0    ->  log(p)   = -inf  (Kötü tahmin)

### Egzersiz 1.4: Cross-Entropy Hesaplama (Manuel)

**Görev:** Ortalama negative log probability hesaplayarak cross-entropy'i bulun

In [ ]:
# Adım 1: Negative log probability hesapla
neg_log_probas = -log_probas
print("Negative log probabilities:", neg_log_probas)

# Adım 2: Ortalama al
avg_neg_log_probas = torch.mean(neg_log_probas)
print(f"\nManuel Cross-Entropy Loss: {avg_neg_log_probas:.4f}")

# Alternatif: -ortalama log prob
avg_log_probas = torch.mean(log_probas)
cross_entropy_manual = -avg_log_probas
print(f"Alternatif hesaplama: {-avg_log_probas:.4f}")

### Egzersiz 1.5: PyTorch cross_entropy ile Doğrulama

**Görev:** PyTorch'un built-in fonksiyonuyla karşılaştırın

In [ ]:
# PyTorch cross_entropy fonksiyonunu kullan
# Not: cross_entropy flat logits ve flat targets bekler
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

loss_pytorch = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(f"PyTorch Cross-Entropy Loss: {loss_pytorch:.4f}")

# Karşılaştırma
print(f"\nFark: {abs(loss_pytorch - avg_neg_log_probas):.6f}")

## Egzersiz 2: Perplexity Hesaplama

**Hedef:** Perplexity'nin ne olduğunu ve nasıl hesaplandığını anlamak

Perplexity = e^(cross-entropy loss)

Intuition: Modelin her adımda kaç kelime arasında seçim yaptığını gösterir

In [ ]:
# Perplexity hesapla
perplexity = torch.exp(loss_pytorch)
print(f"Cross-Entropy Loss: {loss_pytorch:.4f}")
print(f"Perplexity: {perplexity:.2f}")

# Yorumlama:
# Perplexity = 1  -> Mükemmel (her zaman doğru kelimeyi biliyor)
# Perplexity = vocab_size -> Rastgele tahmin
# Bu model eğitimsiz olduğu için yüksek perplexity

### Egzersiz 2.1: Farklı Loss Değerleri için Perplexity Karşılaştırması

In [ ]:
# Farklı loss değerleri için perplexity
loss_values = [0.5, 1.0, 2.0, 5.0, 10.0]
print("Loss -> Perplexity dönüşümü:")
print("-" * 30)
for loss in loss_values:
    ppl = math.exp(loss)
    print(f"Loss: {loss:5.1f} -> Perplexity: {ppl:12.2f}")

# Vocab size ile karşılaştırma
print(f"\nVocabulary size: {GPT_CONFIG_124M['vocab_size']}")
print(f"Rastgele tahmin perplexity: ~{GPT_CONFIG_124M['vocab_size']//2}")

## Egzersiz 3: Training ve Validation Loss Hesaplama

**Hedef:** Gerçek bir veri seti üzerinde loss hesaplama fonksiyonları yazmak

In [ ]:
# Veri setini yükle
import os
import requests

file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

print(f"Veri uzunluğu: {len(text_data)} karakter")

In [ ]:
# Tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
total_tokens = len(tokenizer.encode(text_data))
print(f"Toplam token sayısı: {total_tokens}")

In [ ]:
# Train/Validation split
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

print(f"Eğitim verisi: {len(train_data)} karakter")
print(f"Doğrulama verisi: {len(val_data)} karakter")

In [ ]:
# DataLoader oluştur
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

### Egzersiz 3.1: calc_loss_batch Fonksiyonu

**Görev:** Bir batch için loss hesaplayan fonksiyonu tamamlayın

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    """
    Bir batch için cross-entropy loss hesapla
    
    Args:
        input_batch: Modele giris token ID'leri
        target_batch: Hedef token ID'leri
        model: GPT modeli
        device: Hesaplama cihazı (cuda/cpu)
    
    Returns:
        Loss degeri
    """
    # TODO: Device'a tası
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    
    # TODO: Forward pass
    logits = model(input_batch)
    
    # TODO: Cross-entropy loss hesapla (flatten gerekli)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), 
        target_batch.flatten()
    )
    
    return loss

# Test et
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Bir batch alalım
for input_batch, target_batch in train_loader:
    break

loss = calc_loss_batch(input_batch, target_batch, model, device)
print(f"Batch loss: {loss.item():.4f}")

### Egzersiz 3.2: calc_loss_loader Fonksiyonu

**Görev:** Bir data loader'daki tüm batch'lerin ortalama loss'unu hesaplayan fonksiyonu tamamlayın

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    """
    Data loader'daki ortalama loss'u hesapla
    
    Args:
        data_loader: Veri yükleyici
        model: GPT modeli
        device: Hesaplama cihazı
        num_batches: Hesaplanacak max batch sayısı (None = tümü)
    
    Returns:
        Ortalama loss
    """
    total_loss = 0.
    
    # TODO: Data loader boşsa NaN döndür
    if len(data_loader) == 0:
        return float("nan")
    
    # Batch sayısını belirle
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    
    # Loss hesapla
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    
    # Ortalama al
    return total_loss / num_batches

# Test et
model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")
print(f"\nTraining Perplexity: {math.exp(train_loss):.2f}")
print(f"Validation Perplexity: {math.exp(val_loss):.2f}")

## Egzersiz 4: Loss ve Perplexity Yorumlama

**Soru:** Aşağıdaki senaryoları yorumlayın:

1. Eğitim loss'u düşük, validation loss'u yüksek:
2. Eğitim loss'u ve validation loss'u ikisi de yüksek:
3. Perplexity = 50257 (vocab size) ne anlama gelir?
4. Perplexity = 1.0 ne anlama gelir?

### Cevaplar:

1. **Overfitting:** Model eğitim verisini ezberlemiş, ancak genelleme yapamıyor
2. **Underfitting:** Model henüz veriyi öğrenememiş, daha fazla eğitim gerekli
3. **Rastgele tahmin:** Model her kelimeyi sözlükteki herhangi bir kelime olarak görüyor (en kötü durum)
4. **Mükemmel tahmin:** Model her zaman doğru kelimeyi biliyor (imkansız durum)

## Egzersiz 5: Pratik Uygulama - Loss Monitoring

Basit bir eğitim döngüsü yazarak loss ve perplexity'nin nasıl değiştiğini gözlemleyin:

In [ ]:
# Basit eğitim (sadece birkaç adım)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004)

print("Basit Eğitim Döngüsü - Loss ve Perplexity Takibi")
print("=" * 60)

num_steps = 20
for step, (input_batch, target_batch) in enumerate(train_loader):
    if step >= num_steps:
        break
    
    # Forward pass
    optimizer.zero_grad()
    loss = calc_loss_batch(input_batch, target_batch, model, device)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Her 5 adımda bir loss ve perplexity raporla
    if step % 5 == 0:
        perplexity = math.exp(loss.item())
        print(f"Step {step:3d}: Loss = {loss.item():.4f}, Perplexity = {perplexity:.2f}")

print("=" * 60)

## Özet

Bu egzersizlerde şunları öğrendik:

1. **Cross-Entropy Loss:** Modelin tahminlerinin gerçek hedeflerden ne kadar farklı olduğunu ölçer
   - Hesaplama: -log(doğru_kelime_olasılığı) ortalaması
   
2. **Perplexity:** Cross-entropy'in üstel dönüşümü
   - Yorum: Modelin her adımda kaç kelime arasında "kararsız" olduğu
   - Düşük perplexity = Daha iyi model
   
3. **Pratik:** Gerçek veri üzerinde loss hesaplama ve monitoring